# YOLOv8 small: Vehicle GPU 0 + Traffic Light GPU 1

Enable Internet, two GPUs and RCLONE_CONFIG_TEXT (Base64). Keep this notebook private. Upload only code files, never OAuth JSON. Fresh training, not resume. Run all cells; the final cell waits for both background workers.


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pyyaml"], check=True)
subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True)
subprocess.run(["sudo", "apt-get", "install", "-y", "-qq", "rclone"], check=True)


In [ ]:
from pathlib import Path

CODE_ROOT = Path("/kaggle/input/datasets/v5nhdf/yolov8/kaggle")
VEHICLE_ROOT = "/kaggle/input/datasets/v5nhdf/yolov8/Vehicle363-v1/data/Vehicle363-v1"
LIGHT_ROOT = "/kaggle/input/datasets/v5nhdf/yolov8/TrafficLight363-v1/data/TrafficLight363-v1"
subprocess.run([
    sys.executable, str(CODE_ROOT / "prepare_configs.py"),
    "--vehicle-root", VEHICLE_ROOT, "--traffic-light-root", LIGHT_ROOT
], check=True)


In [ ]:
import base64
import configparser
import os
from kaggle_secrets import UserSecretsClient

config_path = "/tmp/project363_rclone.conf"
configuration = base64.b64decode(
    UserSecretsClient().get_secret("RCLONE_CONFIG_TEXT").strip(), validate=True
).decode("utf-8")
parsed = configparser.ConfigParser(interpolation=None)
parsed.read_string(configuration)
if not parsed.has_section("gdrive2") or parsed["gdrive2"].get("type") != "drive":
    raise ValueError("Secret must contain a gdrive2 Drive configuration")
descriptor = os.open(config_path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
os.fchmod(descriptor, 0o600)
with os.fdopen(descriptor, "w", encoding="utf-8") as handle:
    handle.write(configuration)
del configuration, parsed
print("Secret configured without displaying credentials.")


In [ ]:
subprocess.run([
    sys.executable, "-u", str(CODE_ROOT / "train_drive.py"), "launch",
    "--variant", "small",
    "--rclone-config", config_path,
    "--destination", "gdrive2:Project363/yolov8",
    "--wait"
], check=True)
